# Inference Masterclass I: Quantization/Harness-Es

**Autor y responsable del repositorio:** [Angel Galvis](https://github.com/angelgalvisc) · [LinkedIn](https://www.linkedin.com/in/angelgalvisc/)

## Resumen

Los servicios de inferencia de los laboratorios de frontera han acelerado la adopción empresarial de la inteligencia artificial y han hecho accesibles capacidades extraordinarias mediante una API. Este modelo de consumo, sin embargo, mantiene los pesos, el runtime, la asignación de cómputo y la evolución del modelo bajo el control del proveedor. Para organizaciones con grandes volúmenes de inferencia, los modelos de pesos abiertos crean una alternativa estratégica: permiten especializar, cuantizar y desplegar la inteligencia sobre infraestructura propia o dedicada, dimensionada para cada carga de trabajo. El modelo, su costo operativo y su evolución pasan así a ser decisiones de arquitectura que la organización puede controlar.

Este notebook explora esa oportunidad mediante Alchemist, un modelo agéntico de 4 mil millones de parámetros cuantizado a 4 bits y reducido a aproximadamente 2,4 GB. La demostración recorre su instalación y ejecución en una GPU de Kaggle, compara su comportamiento con el modelo de precisión completa y examina hasta qué punto la cuantización conserva sus capacidades semánticas y de resolución de tareas.

A partir de los comportamientos observados, se construye un harness minimalista que incorpora contratos de salida, límites de ejecución, validación independiente, recuperación acotada y operaciones deterministas. El modelo interpreta la solicitud y propone una acción estructurada; el validador comprueba que cumpla el contrato; y el ejecutor aplica las reglas y cálculos que requieren exactitud. Las trazas conservadas permiten observar cada transición y reconstruir qué propuso el modelo, qué aceptó el sistema y qué operación se ejecutó.

El resultado es una arquitectura híbrida: los modelos frontera permanecen disponibles para problemas abiertos, mientras modelos especializados de pesos abiertos pueden atender cargas repetibles, privadas y de alto volumen. Esta transición hacia la **inteligencia autoalojada (*self-hosted intelligence*)** no consiste simplemente en ejecutar un modelo más pequeño, sino en convertirlo en una unidad de inteligencia auditable, adaptable y desplegable sobre recursos que la organización puede dimensionar y gobernar.

¡Bienvenido! En este notebook instalarás y ejecutarás por primera vez [Agent A1 Alchemist 4-bit](https://huggingface.co/angelgalvisc/agent-a1-alchemist-4bit) en una GPU de Kaggle.

## ¿Qué es Alchemist?

Alchemist es una versión compacta, cuantizada y preparada para MLX de un modelo agéntico de 4.000 millones de parámetros. Su linaje es:

1. [Qwen3.5-4B](https://huggingface.co/Qwen/Qwen3.5-4B), desarrollado por el equipo Qwen de Alibaba, aporta el modelo de lenguaje base.
2. [Agents-A1-4B](https://huggingface.co/InternScience/Agents-A1-4B), desarrollado por InternScience, parte de Qwen3.5-4B e incorpora entrenamiento agéntico sobre trayectorias de uso de herramientas.
3. [Agent A1 Alchemist 4-bit](https://huggingface.co/angelgalvisc/agent-a1-alchemist-4bit) fue cuantizado y empaquetado por **DataStrat** a partir de Agents-A1-4B.

### ¿Cómo cuantizó Datastrat el modelo?

**Datastrat desarrolló Alchemist como una versión MLX compacta de Agents-A1-4B mediante una estrategia de cuantización mixta orientada a preservar su comportamiento agéntico.** El proceso puede resumirse en cinco decisiones técnicas:

1. **Cuantización afín por grupos (*group-wise affine quantization*)**

   Las 248 matrices principales del módulo de lenguaje se dividieron en grupos de 128 pesos. Cada grupo se representa mediante valores de 4 bits y conserva su propia escala y punto cero. Esto permite adaptar la representación numérica al rango local de cada grupo, en lugar de aplicar una única escala a toda la matriz.

2. **Reescalado de activaciones por canal (*per-channel activation rescaling*)**

   Antes del redondeo, Datastrat reescaló 216 de las 248 matrices. Los pesos asociados a canales con activaciones de mayor magnitud se amplificaron para que el redondeo a 4 bits conservara mejor su información.

3. **Compensación integrada en capas vecinas (*scale folding*)**

   El reescalado anterior se compensó dividiendo las activaciones por el mismo factor. Esa operación inversa se integró algebraicamente en la capa vecina, de manera que el modelo conserva el mismo cálculo antes del redondeo sin añadir operaciones durante la inferencia ni aumentar el tamaño del archivo. El efecto práctico es reducir el error de cuantización en los canales más sensibles.

4. **Asignación selectiva de precisión (*mixed-precision allocation*)**

   No todos los componentes se comprimieron de la misma manera:

   - Las 248 proyecciones principales se almacenaron nominalmente a 4 bits.
   - La tabla de vocabulario compartida por la entrada y la salida se conservó a 6 bits, porque las pruebas mostraron una degradación marcada por debajo de esa precisión.
   - Las normalizaciones permanecieron en BF16.
   - Las 32 matrices de salida de atención se cuantizaron sin reescalado, porque no existe una capa lineal anterior adecuada en la cual integrar la compensación.

5. **Conservación completa del rango (*no range clipping*)**

   Datastrat evaluó el recorte de valores extremos, una técnica que puede mejorar algunas métricas convencionales de texto. Sin embargo, observó que también aumentaba la repetición de llamadas fallidas a herramientas. Por esta razón, Alchemist conserva el rango completo de los pesos y prioriza la estabilidad del comportamiento agéntico.

Aunque se denomina “modelo de 4 bits”, esa expresión es una simplificación:

- Las proyecciones utilizan aproximadamente 4,25 bits por peso al incluir escalas y puntos cero.
- La tabla de vocabulario utiliza aproximadamente 6,25 bits por peso.
- El promedio del módulo de lenguaje es de **4,555 bits por peso**.
- El archivo de pesos resultante ocupa aproximadamente **2,395 GB**.

El resultado es [Agent A1 Alchemist 4-bit](https://huggingface.co/angelgalvisc/agent-a1-alchemist-4bit), un *checkpoint* MLX derivado de [Agents-A1-4B](https://huggingface.co/InternScience/Agents-A1-4B) que reduce sustancialmente el almacenamiento y la memoria requeridos, procurando conservar las capacidades agénticas del modelo original.

En las siguientes secciones configuraremos el *backend* CUDA de MLX, confirmaremos que Kaggle asignó una GPU NVIDIA compatible, cargaremos el modelo directamente desde Hugging Face y generaremos su primera respuesta. Esta instalación funcional es la base para utilizar Alchemist con el Alchemist-RLM Harness.

## 1. Configurar la sesión de Kaggle

Antes de ejecutar el notebook, abre **Settings → Accelerator** y selecciona **GPU T4 ×2**. Mantén **Internet on** para descargar el modelo desde Hugging Face.

> Kaggle puede mostrar dos GPU T4, aunque esta introducción utiliza la GPU predeterminada para ejecutar un solo modelo.

## 2. Verificar la GPU e instalar MLX para CUDA

`mlx-lm` permite cargar modelos y generar texto. En Linux con una GPU NVIDIA, MLX también necesita su *backend* CUDA. El entorno T4 actual de Kaggle utiliza un controlador compatible con CUDA 13; por eso fijamos las versiones de ambos paquetes a las utilizadas en esta masterclass.

In [1]:
# Display the NVIDIA GPUs assigned to this Kaggle session.
!nvidia-smi

# Install reproducible versions of MLX with CUDA support and MLX-LM.
%pip install -q --upgrade "mlx[cuda13]==0.32.1" "mlx-lm==0.31.3"

Wed Aug 19 09:56:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Confirmar que MLX utiliza la GPU

La siguiente celda registra las versiones instaladas y verifica que MLX haya seleccionado un dispositivo CUDA. Debes ver `CUDA available: True` y un dispositivo como `Device(gpu, 0)`.

In [2]:
from importlib.metadata import version
import mlx.core as mx
from mlx_lm import load, generate

print("MLX version:", version("mlx"))
print("MLX-LM version:", version("mlx-lm"))
print("CUDA available:", mx.cuda.is_available())
print("Active device:", mx.default_device())

MLX version: 0.32.1
MLX-LM version: 0.31.3
CUDA available: True
Active device: Device(gpu, 0)


## 4. Cargar Alchemist

MLX-LM descarga el *checkpoint* desde su [repositorio en Hugging Face](https://huggingface.co/angelgalvisc/agent-a1-alchemist-4bit) e inicializa el *tokenizer* y el modelo.

Los pesos permanecen cuantizados a 4 bits. Como las GPU Tesla T4 están optimizadas para FP16 y no ofrecen ejecución nativa BF16 en sus *tensor cores*, `set_dtype(mx.float16)` selecciona FP16 para los parámetros y operaciones de punto flotante. Este es el modo de ejecución compatible con esta configuración de Kaggle.

In [3]:
# Download and load Alchemist directly from Hugging Face.
model, tokenizer = load("angelgalvisc/agent-a1-alchemist-4bit")

# Keep the checkpoint quantized while using T4-compatible FP16 operations.
model.set_dtype(mx.float16)

print("Alchemist is loaded and ready.")

You are using a model of type qwen3_5 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


Alchemist is loaded and ready.


## 5. Generar la primera respuesta

Formateamos la conversación con el *chat template* de Alchemist y desactivamos la traza de pensamiento opcional para obtener una presentación breve. `verbose=True` también informa la velocidad de procesamiento del *prompt*, la velocidad de generación y el uso máximo de memoria de la GPU.

In [4]:
messages = [
    {"role": "user", "content": "Say hello briefly and introduce yourself."}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

response = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=96,
    verbose=True,
)

print("\nAlchemist response:")
print(response)

Hello! I'm Agent-A1, a deep research assistant built by Datastrat. I'm here to help you with any questions or tasks you have. How can I assist you today?
Prompt: 378 tokens, 4.399 tokens-per-sec
Generation: 40 tokens, 34.140 tokens-per-sec
Peak memory: 3.579 GB

Alchemist response:
Hello! I'm Agent-A1, a deep research assistant built by Datastrat. I'm here to help you with any questions or tasks you have. How can I assist you today?


## 6. Una segunda pregunta guiada: comprender la cuantización

Esta segunda interacción introduce la idea central del *checkpoint* que ejecuta Alchemist. La pregunta es deliberadamente sencilla: qué es la cuantización, por qué un modelo de 4 bits puede utilizar menos memoria que uno de 16 o 32 bits y cuál puede ser la principal desventaja de reducir la precisión. Solicitar un solo párrafo breve mantiene la respuesta clara y facilita compararla con tu propia ejecución.

In [5]:
quantization_question = (
    "In simple English, what is model quantization? "
    "Why can a 4-bit model use less memory than a 16-bit or 32-bit model, "
    "and what trade-off can lower precision introduce? "
    "Answer in one short paragraph."
)

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": quantization_question}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

quantization_answer = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=384,
    verbose=True,
)

print("\nAlchemist's explanation:")
print(quantization_answer)

Model quantization is a technique that reduces the number of bits used to represent each numerical value in a machine learning model, typically converting high-precision weights (like 32-bit floats) into lower precision formats (like 4-bit integers). A 4-bit model uses less memory because each weight is stored with only 4 bits instead of 16 or 32, which drastically cuts the total storage size—since 4 bits represent a much smaller range of values, the data requires far fewer bytes to store. However, this lower precision introduces a trade-off: while the model becomes more efficient and faster to run, it may lose some accuracy because the reduced number of bits limits the granularity of the numerical representations, potentially causing slight degradation in performance compared to the original high-precision model.
Prompt: 419 tokens, 140.735 tokens-per-sec
Generation: 162 tokens, 34.874 tokens-per-sec
Peak memory: 3.685 GB

Alchemist's explanation:
Model quantization is a technique tha

## 7. Observar el razonamiento en una tarea de conteo exacto

En esta tercera interacción activamos el modo de pensamiento para observar cómo Alchemist aborda una tarea pequeña que exige descomposición y verificación cuidadosas. La frase está inspirada en Star Wars y contiene varias apariciones de la misma letra. El modo de pensamiento utiliza los parámetros de muestreo recomendados para Alchemist, permitiéndole concluir su razonamiento sin quedar atrapado en una verificación *greedy* repetitiva. Una traza visible resulta útil para inspeccionar el proceso, pero debe entenderse como una explicación generada por el modelo y no como un registro garantizado de todos sus cálculos internos.

In [6]:
from mlx_lm.sample_utils import make_logits_processors, make_sampler

reasoning_question = (
    "Count how many times the letter 'r' appears in the exact sentence below, "
    "ignoring uppercase and lowercase. Think briefly: count word by word, verify the "
    "arithmetic once, then give the final total.\n\n"
    "R2-D2 races through space as rebels search for Vader."
)

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": reasoning_question}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

reasoning_response = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=4096,
    sampler=make_sampler(temp=0.85, top_p=0.95, top_k=20),
    logits_processors=make_logits_processors(
        presence_penalty=1.1,
        presence_context_size=100000,
    ),
    verbose=True,
)

print("\nAlchemist's reasoning and final answer:")
print(reasoning_response)

Thinking Process:

1.  **Analyze the Request:**
    *   Task: Count the occurrences of the letter 'r' (case-insensitive) in the provided sentence.
    *   Sentence: "R2-D2 races through space as rebels search for Vader."
    *   Instructions: Think briefly, count word by word, verify arithmetic once, then give the final total.
    *   Role: Agent-A1 (The Alchemist).
    *   Constraints: No tools needed for this simple task (Daily Chat & Simple Questions category). Respond directly and naturally.

2.  **Analyze the Sentence:**
    *   Sentence: "R2-D2 races through space as rebels search for Vader."

3.  **Process - Word by Word:**
    *   Word 1: "R2-D2" -> Contains 'R'. Count = 1. (Note: The prompt says ignore case, so 'R' counts. '2' and '-' are not letters).
    *   Word 2: "races" -> Contains 'r', 'r'? Let's check. r-a-c-e-s. One 'r' at the start. Count = 1. Total so far = 2.
    *   Word 3: "through" -> t-h-r-o-u-g-h. One 'r'. Count = 1. Total so far = 3.
    *   Word 4: "space" -

## 8. De la salida del modelo al resultado del sistema

### Un Completion-Recovery Harness mínimo

Un modelo puede resolver una tarea dentro de su traza de razonamiento y aun así no producir una respuesta que una aplicación pueda entregar. Esta sección demuestra esa diferencia mediante un problema original de seguimiento de estados inspirado en la familia de tareas de razonamiento *Tracking Shuffled Objects*.

La ejecución base utiliza deliberadamente decodificación *greedy*: el mismo modelo, *prompt*, entorno y dispositivo siguen la misma ruta de decodificación. En la ejecución de referencia, Alchemist sigue correctamente los cinco intercambios, pero verifica su trabajo repetidamente hasta agotar el presupuesto de 4.096 tokens dentro del bloque de razonamiento. Los pesos no se modifican. En su lugar, una pequeña política externa detecta la terminación incompleta y abre condicionalmente un paso limpio para producir la respuesta.

La pregunta está escrita en español para demostrar que la política es independiente del idioma de la tarea.

In [7]:
# Keep the exact tested prompt byte-for-byte: greedy decoding is sensitive not
# only to meaning, but also to punctuation and layout.
state_tracking_question = (
    "Cinco droides llevan un objeto diferente cada uno. "
    "Al principio: R2-D2 lleva el mapa estelar; C-3PO lleva la llave de acceso; "
    "BB-8 lleva la baliza de rastreo; K-2SO lleva el cilindro de códigos; "
    "Chopper lleva el holocrón. Después intercambian sus objetos en este orden: "
    "1. R2-D2 y BB-8. 2. C-3PO y K-2SO. 3. BB-8 y Chopper. "
    "4. K-2SO y R2-D2. 5. Chopper y C-3PO. Determina qué objeto lleva cada "
    "droide al final. Verifica cuidadosamente cada intercambio y termina con "
    "una correspondencia concisa."
)

print(state_tracking_question)

Cinco droides llevan un objeto diferente cada uno. Al principio: R2-D2 lleva el mapa estelar; C-3PO lleva la llave de acceso; BB-8 lleva la baliza de rastreo; K-2SO lleva el cilindro de códigos; Chopper lleva el holocrón. Después intercambian sus objetos en este orden: 1. R2-D2 y BB-8. 2. C-3PO y K-2SO. 3. BB-8 y Chopper. 4. K-2SO y R2-D2. 5. Chopper y C-3PO. Determina qué objeto lleva cada droide al final. Verifica cuidadosamente cada intercambio y termina con una correspondencia concisa.


![Tabla de estados con el objeto inicial de cada uno de los cinco droides, los cinco intercambios y la correspondencia final.](https://raw.githubusercontent.com/angelgalvisc/alchemist-rlm/c01c09f/docs/assets/masterclass/five-droid-swap-tracking-es.png)

*Figura 1. Seguimiento determinista de la tarea de cinco droides. Cada fila representa el estado completo después de un intercambio. El resultado final coincide con la respuesta entregada por el Completion-Recovery Harness.*


## 9. Contrato formal de ejecución

Sea el primer paso del modelo

$$y_1 = M(x; c_r).$$

La notación tiene cuatro componentes:

- $M$ es el modelo de lenguaje sin modificar.
- $x$ es la pregunta original del usuario.
- $c_r$ es la configuración de razonamiento: pensamiento activado, decodificación *greedy* y un límite de 4.096 tokens.
- $y_1$ es la salida observable completa de ese paso, incluido el texto de razonamiento y los metadatos de terminación.

El harness **no** determina si la correspondencia entre droides y objetos es verdadera. Solo evalúa si la salida puede entregarse estructuralmente. Definimos

$$D(y)=L(y)\land A(y)\land N(y),$$

donde:

- $L(y)$ es verdadero cuando el bloque de razonamiento está completo. Para este adaptador, significa que aparece `</think>`.
- $A(y)$ es verdadero cuando existe contenido de respuesta no vacío después del bloque de razonamiento.
- $N(y)$ es verdadero cuando la generación termina normalmente y no porque se alcanzó el límite de tokens.

Por tanto, $D(y)=1$ significa *entregable*, no necesariamente *correcto*. La corrección y la capacidad de entrega siguen siendo mediciones independientes.

La salida de recuperación acotada es

$$
y_2=M(R(x,y_1);c_f).
$$

La política del harness de entrega es

$$
H_D(M,x)=
\begin{cases}
y_1, & D(y_1)=1,\\[4pt]
y_2, & D(y_1)=0 \land D(y_2)=1,\\[4pt]
\bot, & \text{en cualquier otro caso}.
\end{cases}
$$

Aquí, $H_D$ es el harness de entrega, $R(x,y_1)$ construye una solicitud de recuperación a partir de la pregunta original y del borrador intacto, y $c_f$ es una configuración breve de respuesta con el razonamiento desactivado. La política permite como máximo un paso de recuperación y devuelve un fallo explícito si ese paso tampoco satisface el contrato de entrega.

En términos simples: **Detectar → Orientar → Recuperar**.

## 10. Implementar el harness en Python puro

La implementación evita deliberadamente utilizar un *framework*. `run_step` realiza una solicitud al modelo, `inspect_completion` aplica el contrato estructural y `completion_recovery` controla el segundo paso condicional. El detalle específico del modelo —cómo reconocer el cierre del bloque de razonamiento— queda aislado en el inspector y no disperso por toda la política.

In [8]:
from mlx_lm import stream_generate


def run_step(user_content, *, thinking, max_tokens):
    """Run one greedy model step and retain its termination metadata."""
    formatted_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking,
    )

    fragments = []
    last_event = None
    for event in stream_generate(
        model, tokenizer, prompt=formatted_prompt, max_tokens=max_tokens
    ):
        fragments.append(event.text)
        last_event = event

    if last_event is None:
        raise RuntimeError("The model produced no generation events.")

    return {
        "text": "".join(fragments),
        "thinking": thinking,
        "tokens": last_event.generation_tokens,
        "tokens_per_second": last_event.generation_tps,
        "finish_reason": last_event.finish_reason,
    }


def inspect_completion(step):
    """Measure deliverability, not semantic correctness."""
    text = step["text"]

    if step["thinking"]:
        reasoning_closed = "</think>" in text
        final_content = (
            text.rsplit("</think>", 1)[1].strip() if reasoning_closed else ""
        )
    else:
        reasoning_closed = True
        final_content = text.strip()

    stopped_normally = step["finish_reason"] == "stop"
    deliverable = reasoning_closed and bool(final_content) and stopped_normally

    return {
        "reasoning_closed": reasoning_closed,
        "has_final_content": bool(final_content),
        "stopped_normally": stopped_normally,
        "deliverable": deliverable,
        "final_content": final_content,
    }


def completion_recovery(question, *, reasoning_budget=4096, answer_budget=512):
    """Return a deliverable answer after at most one recovery step."""
    first = run_step(question, thinking=True, max_tokens=reasoning_budget)
    first_check = inspect_completion(first)

    if first_check["deliverable"]:
        return {
            "answer": first_check["final_content"],
            "recovery_used": False,
            "first_step": first,
            "first_check": first_check,
            "recovery_step": None,
            "recovery_check": None,
        }

    recovery_request = f"""{question}

A continuación aparece el borrador bruto de un intento anterior. Puede estar truncado, ser repetitivo o no haber cerrado su bloque de razonamiento. No asumas que su formato es correcto.

--- BORRADOR ---
{first['text']}
--- FIN DEL BORRADOR ---

Responde la pregunta original usando la información válida del borrador. Entrega solamente la correspondencia final, una línea por droide, sin razonamiento, introducción ni conclusión."""

    recovery = run_step(
        recovery_request, thinking=False, max_tokens=answer_budget
    )
    recovery_check = inspect_completion(recovery)

    if not recovery_check["deliverable"]:
        raise RuntimeError(
            "The single permitted recovery step was not deliverable."
        )

    return {
        "answer": recovery_check["final_content"],
        "recovery_used": True,
        "first_step": first,
        "first_check": first_check,
        "recovery_step": recovery,
        "recovery_check": recovery_check,
    }

## 11. Ejecutar el experimento controlado

Esta llamada ejecuta la política completa. El primer paso es la ejecución base del modelo sin asistencia. Si la salida no es entregable, el harness activa exactamente un paso limpio. La ejecución base tarda más que la recuperación porque puede consumir hasta 4.096 tokens.

In [9]:
harness_result = completion_recovery(state_tracking_question)

first = harness_result["first_step"]
check = harness_result["first_check"]

print("MODEL-ONLY STEP")
print("-----------------")
print(f"Generated tokens:     {first['tokens']}")
print(f"Tokens per second:    {first['tokens_per_second']:.2f}")
print(f"Finish reason:        {first['finish_reason']}")
print(f"Reasoning closed:     {check['reasoning_closed']}")
print(f"Final content found:  {check['has_final_content']}")
print(f"Deliverable:          {check['deliverable']}")
print(f"Recovery activated:   {harness_result['recovery_used']}")

# Show enough of the raw draft to make repeated verification visible without
# flooding the notebook. The complete 4,096-token draft remains auditable as
# harness_result["first_step"]["text"].
preview_chars = 1800
print("\nTAIL OF THE RAW REASONING DRAFT")
print("--------------------------------")
print(first["text"][-preview_chars:])

if harness_result["recovery_used"]:
    recovery = harness_result["recovery_step"]
    recovery_check = harness_result["recovery_check"]
    print("\nRECOVERY STEP")
    print("-------------")
    print(f"Generated tokens:     {recovery['tokens']}")
    print(f"Tokens per second:    {recovery['tokens_per_second']:.2f}")
    print(f"Finish reason:        {recovery['finish_reason']}")
    print(f"Deliverable:          {recovery_check['deliverable']}")

print("\nFINAL DELIVERABLE ANSWER")
print("------------------------")
print(harness_result["answer"])

MODEL-ONLY STEP
-----------------
Generated tokens:     4096
Tokens per second:    31.53
Finish reason:        length
Reasoning closed:     False
Final content found:  False
Deliverable:          False
Recovery activated:   True

TAIL OF THE RAW REASONING DRAFT
--------------------------------
ay, final output generation.

    Wait, I should check the date. Current date: 2026-08-19. This is irrelevant for the puzzle but good to keep in mind if needed. Not needed here.

    Okay, generating response.

    Wait, I should check if there is any ambiguity about "intercambian sus objetos en este orden". Does it mean they swap simultaneously or sequentially? "En este orden" usually implies sequential steps 1, 2, 3... So my sequential assumption is correct.

    Okay, ready.

    Final check on Spanish grammar.
    "Llave de acceso" - correct.
    "Mapa estelar" - correct.
    "Baliza de rastreo" - correct.
    "Cilindro de códigos" - correct.
    "Holocrón" - correct.

    Okay.

    Let's pr

## 12. Qué cambió el harness y qué permaneció igual

El primer paso y el paso de recuperación utilizan los mismos pesos del modelo. El harness no contiene la correspondencia correcta de los droides, no edita el borrador de razonamiento ni realiza calificación semántica. Su contribución es la política de ejecución:

1. conservar el borrador observable completo del modelo;
2. inspeccionar si la salida satisface un contrato de entrega;
3. impedir la terminación cuando el contrato no se satisface;
4. abrir un paso de respuesta acotado con el pensamiento desactivado; y
5. informar explícitamente el fallo, en lugar de repetir indefinidamente, si la recuperación no tiene éxito.

Esta distinción es la enseñanza central: la capacidad del modelo y la confiabilidad del sistema están relacionadas, pero no son idénticas. El modelo puede contener información suficiente para resolver una tarea, mientras que el bucle que lo rodea determina si esa información se convierte en un resultado utilizable.

El experimento demuestra **recuperación de finalización**, no prueba de corrección semántica ni modelado recursivo del lenguaje. Un harness de producción puede incorporar validadores semánticos independientes, herramientas, persistencia o recursión como políticas separadas.

## 13. Cuantización y estabilidad del comportamiento

### Comparación de Alchemist con su modelo original sin cuantizar

[Agents-A1-4B](https://huggingface.co/InternScience/Agents-A1-4B) es el *checkpoint* original de 16 bits del cual se cuantizó [Alchemist 4-bit](https://huggingface.co/angelgalvisc/agent-a1-alchemist-4bit). El linaje evaluado es **Agents-A1-4B (BF16) → cuantización a 4 bits → Alchemist**.

Los archivos de pesos publicados ocupan aproximadamente **9,08 GB** para Agents y **3,06 GB** para Alchemist: una reducción de **66,3 %**, equivalente a un modelo aproximadamente **2,97 veces más pequeño**. En la GPU T4 de Kaggle, Agents se ejecuta en FP16 porque la T4 no ofrece ejecución nativa BF16 en sus *tensor cores*. Esta conversión de ejecución BF16 a FP16 mantiene el modelo original en precisión de 16 bits; no es una cuantización a 4 bits.

La siguiente celda es autocontenida, por lo que esta comparación puede ejecutarse sin repetir las demostraciones anteriores de Alchemist. Utiliza exactamente el mismo problema de seguimiento de estados en español, el modo de pensamiento del *chat template*, decodificación *greedy* y el límite de 4.096 tokens utilizados con Alchemist. Se conserva y muestra la traza completa generada, junto con los metadatos de terminación y la memoria máxima de GPU.

In [10]:
# Self-contained Agents-A1-4B comparison: only this cell needs to be run.
%pip install -q --upgrade "mlx[cuda13]==0.32.1" "mlx-lm==0.31.3"

import gc
import mlx.core as mx
from mlx_lm import load, stream_generate
from mlx_lm.sample_utils import make_sampler

for name in ("model", "tokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
mx.clear_cache()
mx.reset_peak_memory()

agents_model, agents_tokenizer = load("InternScience/Agents-A1-4B")
# T4-compatible 16-bit execution; this does not quantize the parent checkpoint.
agents_model.set_dtype(mx.float16)
mx.eval(agents_model.parameters())
mx.reset_peak_memory()

state_tracking_question = (
    "Cinco droides llevan un objeto diferente cada uno. "
    "Al principio: R2-D2 lleva el mapa estelar; C-3PO lleva la llave de acceso; "
    "BB-8 lleva la baliza de rastreo; K-2SO lleva el cilindro de códigos; "
    "Chopper lleva el holocrón. Después intercambian sus objetos en este orden: "
    "1. R2-D2 y BB-8. 2. C-3PO y K-2SO. 3. BB-8 y Chopper. "
    "4. K-2SO y R2-D2. 5. Chopper y C-3PO. Determina qué objeto lleva cada "
    "droide al final. Verifica cuidadosamente cada intercambio y termina con "
    "una correspondencia concisa."
)

agents_prompt = agents_tokenizer.apply_chat_template(
    [{"role": "user", "content": state_tracking_question}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

parts = []
last_event = None
for event in stream_generate(
    agents_model,
    agents_tokenizer,
    prompt=agents_prompt,
    max_tokens=4096,
    sampler=make_sampler(temp=0.0),
):
    parts.append(event.text)
    last_event = event

if last_event is None:
    raise RuntimeError("Agents-A1-4B produced no generation events.")

agents_text = "".join(parts)
reasoning_closed = "</think>" in agents_text
final_text = agents_text.split("</think>", 1)[1].strip() if reasoning_closed else ""

print("AGENTS-A1-4B — ORIGINAL 16-BIT PARENT")
print("--------------------------------------")
print(f"Generated tokens:     {last_event.generation_tokens}")
print(f"Tokens per second:    {last_event.generation_tps:.2f}")
print(f"Peak GPU memory:      {last_event.peak_memory:.3f} GB")
print(f"Finish reason:        {last_event.finish_reason}")
print(f"Reasoning closed:     {reasoning_closed}")
print(f"Final content found:  {bool(final_text)}")

print("\nFULL GENERATED TRACE")
print("--------------------")
print(agents_text)

print("\nCONTROLLED COMPARISON")
print("---------------------")
print("Alchemist 4-bit: 3.06 GB weights | 4096 tokens | length | no deliverable")
print(
    f"Agents 16-bit:   9.08 GB weights | {last_event.peak_memory:.3f} GB peak | "
    f"{last_event.generation_tokens} tokens | {last_event.finish_reason} | "
    f"{'deliverable' if final_text else 'no deliverable'}"
)

Note: you may need to restart the kernel to use updated packages.


You are using a model of type qwen3_5 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


AGENTS-A1-4B — ORIGINAL 16-BIT PARENT
--------------------------------------
Generated tokens:     2402
Tokens per second:    26.25
Peak GPU memory:      9.678 GB
Finish reason:        stop
Reasoning closed:     True
Final content found:  True

FULL GENERATED TRACE
--------------------
Thinking Process:

1.  **Analyze the Request:**
    *   **Task:** Determine which object each droid carries at the end after a series of exchanges.
    *   **Initial State:**
        *   R2-D2: Star Map (Mapa estelar)
        *   C-3PO: Access Key (Llave de acceso)
        *   BB-8: Tracking Beacon (Baliza de rastreo)
        *   K-2SO: Code Cylinder (Cilindro de códigos)
        *   Chopper: Holocron (Holocrón)
    *   **Exchanges (in order):**
        1.  R2-D2 and BB-8
        2.  C-3PO and K-2SO
        3.  BB-8 and Chopper
        4.  K-2SO and R2-D2
        5.  Chopper and C-3PO
    *   **Output Requirement:** Carefully verify each exchange and end with a concise correspondence.
    *   **Language:

### Interpretación

Bajo el mismo contrato de decodificación, el modelo padre de 16 bits completó la tarea en **2.402 tokens a 26,25 tokens/s**, se detuvo de forma natural, cerró su bloque de razonamiento y devolvió una respuesta final. Su pico medido de memoria GPU fue de **9,678 GB**. El primer intento sin asistencia de Alchemist alcanzó el límite de 4.096 tokens sin cerrar el bloque de razonamiento ni exponer una respuesta final; el paso de recuperación produjo después una respuesta entregable en 47 tokens.

Esta única tarea controlada **no** demuestra que la cuantización por sí sola causara el bucle: el empaquetado del checkpoint, la ejecución numérica y la sensibilidad de la tarea también pueden influir. Sí muestra una diferencia de comportamiento relevante, compatible con que la inferencia de precisión reducida afecte la estabilidad de terminación. Por tanto, el Harness de recuperación de finalización es útil en el límite del sistema: convierte una ejecución que no entrega respuesta en una respuesta final acotada. El harness no reconstruye los pesos de 16 bits ni afirma restaurar todos los comportamientos del modelo padre.


## Instalación y demostración del harness completadas

Alchemist funciona con MLX sobre una GPU T4 de Kaggle. Has probado la generación directa, inspeccionado una traza de razonamiento opcional, ejecutado un Completion-Recovery Harness mínimo y comparado el modelo cuantizado con su modelo original de 16 bits. El modelo y el *tokenizer* pueden conectarse ahora al Alchemist-RLM Harness completo para flujos de trabajo de contexto extenso y tareas agénticas.

## 14. Ejemplo aplicado: de una pregunta cotidiana a un resultado verificable

Una persona no formula filtros, operaciones ni consultas estructuradas. Simplemente pregunta:

> **Oye, ¿cuánto me gasté en servicios públicos el mes pasado?**

La pregunta se conserva literalmente en las dos rutas de ejecución. Los pesos del modelo y el conjunto de transacciones tampoco cambian; lo que cambia es la política de ejecución que los rodea.

| ID | Fecha | Movimiento | Valor | Estado |
|---|---|---|---:|---|
| `M001` | 02 jul | SUPERMERCADO ÉXITO | $84.650 | Aprobada |
| `M002` | 03 jul | ENEL COLOMBIA | $167.420 | Aprobada |
| `M003` | 03 jul | PSE ENEL COLOMBIA | $167.420 | Rechazada |
| `M004` | 05 jul | UBER | $18.900 | Aprobada |
| `M005` | 07 jul | VANTI GAS NATURAL | $61.380 | Aprobada |
| `M006` | 08 jul | PAN PA' YA | $16.700 | Aprobada |
| `M007` | 10 jul | CLARO HOGAR | $112.900 | Aprobada |
| `M008` | 12 jul | ACUEDUCTO DE BOGOTÁ | $96.750 | Aprobada |
| `M009` | 12 jul | PSE ACUEDUCTO BOGOTÁ | $96.750 | Rechazada |
| `M010` | 14 jul | RECARGA NEQUI | $30.000 | Aprobada |
| `M011` | 17 jul | RESTAURANTE CREPES | $57.800 | Aprobada |
| `M012` | 19 jul | VANTI GAS NATURAL | $61.380 | Rechazada |
| `M013` | 21 jul | NETFLIX | $26.900 | Aprobada |
| `M014` | 23 jul | ENVÍO A CAMILA | $75.000 | Aprobada |
| `M015` | 25 jul | TRANSMILENIO | $20.000 | Aprobada |
| `M016` | 28 jul | FARMATODO | $43.250 | Aprobada |
| `M017` | 30 jul | TIENDAS D1 | $71.320 | Aprobada |

### Contrato de ejecución

Sea:

- $q$ la pregunta de la persona;
- $T$ el conjunto de transacciones;
- $M$ el modelo local;
- $\Sigma$ el conjunto de operaciones, categorías y estados permitidos;
- $p$ un plan propuesto por el modelo;
- $V(p)$ el validador del plan; y
- $E(T,p)$ el ejecutor determinista.

Una respuesta directa pide al modelo interpretar, filtrar y calcular dentro de una sola generación:

$$
y_{\mathrm{directo}} = M(q,T).
$$

El harness primero solicita un plan:

$$
p_1=M(q,\Sigma).
$$

Su política acotada es

$$
H_V(M,q,T)=
\begin{cases}
E(T,p_1), & V(p_1)=1,\\[4pt]
E(T,p_2), & V(p_1)=0 \land V(p_2)=1,\\[4pt]
\bot, & \text{en otro caso},
\end{cases}
$$

donde

$$
p_2=M\bigl(R(q,p_1),\Sigma\bigr).
$$

$H_V$ es el harness de ejecución validada. $R$ representa una única solicitud acotada de reparación. $\bot$ significa que el sistema falla explícitamente en lugar de entregar una respuesta no verificable. La salida registrada abajo indica si se activó la ruta acotada de reparación.

La siguiente celda es autocontenida. Carga Alchemist una sola vez, envía exactamente la misma pregunta natural por ambas rutas, conserva las dos salidas observables y ejecuta en Python únicamente la aritmética validada.


In [3]:
# Self-contained applied example: run only this cell.
%pip install -q --upgrade "mlx[cuda13]==0.32.1" "mlx-lm==0.31.3"

import gc
import json
import re
import mlx.core as mx
from mlx_lm import load, generate

# Release another model if this cell is run in an existing session.
for name in ("model", "tokenizer", "agents_model", "agents_tokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
mx.clear_cache()

model, tokenizer = load("angelgalvisc/agent-a1-alchemist-4bit")
model.set_dtype(mx.float16)
mx.eval(model.parameters())

question = "Oye, ¿cuánto me gasté en servicios públicos el mes pasado?"
transactions = [
    {"id":"M001","date":"2026-07-02","merchant":"SUPERMERCADO ÉXITO","amount":84650,"status":"approved","category":"groceries"},
    {"id":"M002","date":"2026-07-03","merchant":"ENEL COLOMBIA","amount":167420,"status":"approved","category":"electricity"},
    {"id":"M003","date":"2026-07-03","merchant":"PSE ENEL COLOMBIA","amount":167420,"status":"rejected","category":"electricity"},
    {"id":"M004","date":"2026-07-05","merchant":"UBER","amount":18900,"status":"approved","category":"transport"},
    {"id":"M005","date":"2026-07-07","merchant":"VANTI GAS NATURAL","amount":61380,"status":"approved","category":"gas"},
    {"id":"M006","date":"2026-07-08","merchant":"PAN PA' YA","amount":16700,"status":"approved","category":"restaurants"},
    {"id":"M007","date":"2026-07-10","merchant":"CLARO HOGAR","amount":112900,"status":"approved","category":"internet"},
    {"id":"M008","date":"2026-07-12","merchant":"ACUEDUCTO DE BOGOTÁ","amount":96750,"status":"approved","category":"water"},
    {"id":"M009","date":"2026-07-12","merchant":"PSE ACUEDUCTO BOGOTÁ","amount":96750,"status":"rejected","category":"water"},
    {"id":"M010","date":"2026-07-14","merchant":"RECARGA NEQUI","amount":30000,"status":"approved","category":"top_up"},
    {"id":"M011","date":"2026-07-17","merchant":"RESTAURANTE CREPES","amount":57800,"status":"approved","category":"restaurants"},
    {"id":"M012","date":"2026-07-19","merchant":"VANTI GAS NATURAL","amount":61380,"status":"rejected","category":"gas"},
    {"id":"M013","date":"2026-07-21","merchant":"NETFLIX","amount":26900,"status":"approved","category":"subscriptions"},
    {"id":"M014","date":"2026-07-23","merchant":"ENVÍO A CAMILA","amount":75000,"status":"approved","category":"transfers"},
    {"id":"M015","date":"2026-07-25","merchant":"TRANSMILENIO","amount":20000,"status":"approved","category":"transport"},
    {"id":"M016","date":"2026-07-28","merchant":"FARMATODO","amount":43250,"status":"approved","category":"health"},
    {"id":"M017","date":"2026-07-30","merchant":"TIENDAS D1","amount":71320,"status":"approved","category":"groceries"},
]

allowed_operations = {"sum", "count", "list"}
allowed_statuses = {"approved", "rejected", "reversed"}
allowed_categories = {tx["category"] for tx in transactions}

def prompt_for(content):
    return tokenizer.apply_chat_template(
        [{"role":"user", "content":content}], tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )

def ask(content, max_tokens):
    return generate(model, tokenizer, prompt=prompt_for(content),
                    max_tokens=max_tokens, verbose=False).strip()

def ledger_text():
    lines = ["Estas son mis transacciones de julio de 2026:"]
    status_es = {"approved":"Aprobada", "rejected":"Rechazada"}
    for tx in transactions:
        amount = "$" + f'{tx["amount"]:,}'.replace(",", ".")
        lines.append(
            f'{tx["id"]} | {tx["date"]} | {tx["merchant"]} | '
            f'{amount} | {status_es[tx["status"]]}'
        )
    return "\n".join(lines)

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("The model did not return a JSON plan.")
    return json.loads(match.group(0))

def valid(plan):
    filters = plan.get("filters", {})
    return (
        plan.get("operation") in allowed_operations
        and set(filters.get("categories", [])) <= allowed_categories
        and set(filters.get("status", [])) <= allowed_statuses
        and isinstance(filters.get("date_from"), str)
        and isinstance(filters.get("date_to"), str)
    )

def execute(plan):
    filters = plan["filters"]
    selected = [
        tx for tx in transactions
        if filters["date_from"] <= tx["date"] <= filters["date_to"]
        and tx["category"] in filters["categories"]
        and tx["status"] in filters["status"]
    ]
    operation = plan["operation"]
    value = (sum(tx["amount"] for tx in selected) if operation == "sum"
             else len(selected) if operation == "count" else selected)
    return {"value": value, "transactions": selected}

def pesos(value):
    return "$" + f"{value:,}".replace(",", ".")

# Path A: the model interprets, filters and calculates in one generation.
direct_input = ledger_text() + "\n\n" + question
if "direct_answer" not in globals():
    direct_answer = ask(direct_input, 1024)

# Path B: the same question becomes a validated executable plan.
contract = f"""
Convierte una pregunta sobre movimientos financieros en un plan JSON ejecutable.
Hoy es 2026-08-19.
Contrato: {{"operation":"<one operation>","filters":{{"date_from":"YYYY-MM-DD","date_to":"YYYY-MM-DD","categories":["..."],"status":["..."]}}}}
Elige operation como exactamente un valor de ["sum", "count", "list"].
Elige cada status únicamente de ["approved", "rejected", "reversed"].
Categorías disponibles: {sorted(allowed_categories)}.
Política del producto: "servicios públicos" significa electricity, gas y water;
internet y top_up son categorías distintas. Un gasto efectivo debe estar approved.
Devuelve solo el JSON, sin calcular valores ni seleccionar IDs manualmente.
Pregunta: {question}
""".strip()
raw_plan = ask(contract, 512)
plan = extract_json(raw_plan)
repair_used = False

if not valid(plan):
    repair_used = True
    repair = contract + "\nEl plan anterior no cumplió el contrato. Elige un solo valor permitido para operation y devuelve únicamente un plan JSON válido."
    raw_plan = ask(repair, 512)
    plan = extract_json(raw_plan)
if not valid(plan):
    raise RuntimeError("The bounded repair did not produce a valid plan.")

result = execute(plan)
assert question in direct_input and question in contract

print("SAME USER QUESTION IN BOTH PATHS")
print("--------------------------------")
print(question)
print("\nMODEL-ONLY ANSWER")
print("-----------------")
print(direct_answer)
print("\nMODEL PLAN")
print("----------")
print(json.dumps(plan, ensure_ascii=False, indent=2))
print(f"\nContract valid: {valid(plan)}")
print(f"Repair used:   {repair_used}")
print("\nDETERMINISTIC EXECUTION")
print("-----------------------")
for tx in result["transactions"]:
    print(f'✓ {tx["id"]}  {tx["merchant"]:<24} {pesos(tx["amount"]):>10}')
print(f'\nTOTAL: {pesos(result["value"])}')


SAME USER QUESTION IN BOTH PATHS
--------------------------------
Oye, ¿cuánto me gasté en servicios públicos el mes pasado?

MODEL-ONLY ANSWER
-----------------
Para determinar cuánto gastaste en servicios públicos en julio de 2026, necesitamos identificar las transacciones relacionadas con servicios públicos (como luz, agua, gas, teléfono, etc.) y sumar solo las que fueron **aprobadas**.

Aquí están las transacciones que corresponden a servicios públicos:

1. **M002 | 2026-07-03 | ENEL COLOMBIA | $167.420 | Aprobada** → Luz (aprobada)
2. **M003 | 2026-07-03 | PSE ENEL COLOMBIA | $167.420 | Rechazada** → Luz (rechazada, no cuenta)
3. **M005 | 2026-07-07 | VANTI GAS NATURAL | $61.380 | Aprobada** → Gas (aprobada)
4. **M006 | 2026-07-08 | PAN PA' YA | $16.700 | Aprobada** → No es un servicio público (es comida)
5. **M007 | 2026-07-10 | CLARO HOGAR | $112.900 | Aprobada** → Teléfono/Internet (aprobada)
6. **M008 | 2026-07-12 | ACUEDUCTO DE BOGOTÁ | $96.750 | Aprobada** → Agua (aprobada)


### Qué ocurrió en esta corrida de Kaggle

La celda anterior conserva la salida observable completa de las dos rutas. Ambas utilizaron el mismo checkpoint de Alchemist, la misma pregunta en lenguaje natural y el mismo conjunto de transacciones.

| Etapa | Sin harness | Con harness |
|---|---|---|
| Entrada en lenguaje natural | La misma pregunta | La misma pregunta |
| Responsabilidad del modelo | Interpretar, seleccionar y calcular | Producir un plan ejecutable |
| Resultado intermedio observable | Respuesta libre | Plan JSON validable |
| Selección | Incluyó CLARO HOGAR y TRANSMILENIO | Pagos aprobados de energía, gas y agua |
| Aritmética | Realizada por el modelo | Realizada por Python |
| Resultado | **$458.450** | **$325.550** |
| Validación del contrato | No disponible | `True` |
| Reparación | No aplica | No utilizada |

#### Ruta directa

El modelo intentó completar toda la tarea dentro de una sola generación. Excluyó correctamente las transacciones rechazadas, pero clasificó `CLARO HOGAR` y `TRANSMILENIO` como servicios públicos e informó **$458.450**.

#### Ruta con harness

El modelo no calculó el resultado. Produjo un plan que especificó el intervalo de fechas, las categorías `electricity`, `gas` y `water`, y el estado `approved`. El plan superó la validación en el primer intento. Python seleccionó `M002`, `M005` y `M008` y calculó **$325.550**.

```text
MISMA PREGUNTA
      │
      ├── Modelo directo ──→ selección libre ──→ $458.450
      │
      └── Harness ──→ plan JSON ──→ validación ──→ Python ──→ $325.550
```

El plan JSON es una traza operacional observable: registra lo que el sistema decidió ejecutar sin requerir acceso a una cadena de pensamiento privada.

> **El modelo directo produjo una respuesta. El harness produjo un resultado verificable.**


## 15. Bucles de razonamiento gestionables: observabilidad y recuperación con un harness

Un harness puede conservar y utilizar el trabajo valioso de una trayectoria de razonamiento incluso cuando la generación no llega a completarse. Esta sección muestra cómo observar ese estado, identificar resultados intermedios verificables y convertirlos en una decisión controlada del sistema.

Repetimos la misma pregunta, el mismo conjunto de transacciones y el mismo checkpoint de Alchemist bajo una única configuración de razonamiento $c_r$: razonamiento activado, decodificación *greedy* y un máximo de 4.096 tokens por ruta.

> **Oye, ¿cuánto me gasté en servicios públicos el mes pasado?**

La ruta directa pide al modelo interpretar, seleccionar, calcular y responder. La ruta con harness le pide producir un plan ejecutable bajo un contrato explícito. Ambas trazas observables se conservan. El sistema no califica pensamientos individuales; comprueba si una traza contiene un estado intermedio verificable de manera independiente.

Sea $\tau$ una traza observable, $\Sigma$ el contrato del plan, $V(p)$ su validador y $E(T,p)$ el ejecutor determinista. Las dos trazas con configuración equivalente son

$$
\tau_{\mathrm{directa}}=M(q,T;c_r),
\qquad
\tau_{\mathrm{plan}}=M(q,\Sigma;c_r).
$$

El conjunto de candidatos JSON válidos según el esquema y observables en una traza es

$$
C(\tau)=\left\{p\in\operatorname{JSON}(\tau):V(p)=1\right\}.
$$

La recuperación estructural se define como

$$
S(\tau,T)=
\begin{cases}
E(T,p), & C(\tau)=\{p\},\\[4pt]
\bot, & |C(\tau)|\neq 1.
\end{cases}
$$

El harness solo continúa cuando exactamente un candidato observable y distinto satisface el contrato independiente; en cualquier otro caso falla explícitamente.

La siguiente celda es autocontenida para una ejecución nueva. En esta actualización registrada reutiliza la traza directa existente de 4.096 tokens de la sesión activa de Kaggle y ejecuta únicamente la ruta del harness con el mismo presupuesto de 4.096 tokens. No se vuelve a ejecutar ninguna celda anterior.


In [1]:
# Self-contained thinking comparison: run only this cell.
%pip install -q --upgrade "mlx[cuda13]==0.32.1" "mlx-lm==0.31.3"

import gc
import json
from pathlib import Path
import mlx.core as mx
from mlx_lm import load, stream_generate
from mlx_lm.sample_utils import make_sampler

for name in ("model", "tokenizer", "agents_model", "agents_tokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
mx.clear_cache()

model, tokenizer = load("angelgalvisc/agent-a1-alchemist-4bit")
model.set_dtype(mx.float16)
mx.eval(model.parameters())

question = "Oye, ¿cuánto me gasté en servicios públicos el mes pasado?"
transactions = [
    {"id":"M001","date":"2026-07-02","merchant":"SUPERMERCADO ÉXITO","amount":84650,"status":"approved","category":"groceries"},
    {"id":"M002","date":"2026-07-03","merchant":"ENEL COLOMBIA","amount":167420,"status":"approved","category":"electricity"},
    {"id":"M003","date":"2026-07-03","merchant":"PSE ENEL COLOMBIA","amount":167420,"status":"rejected","category":"electricity"},
    {"id":"M004","date":"2026-07-05","merchant":"UBER","amount":18900,"status":"approved","category":"transport"},
    {"id":"M005","date":"2026-07-07","merchant":"VANTI GAS NATURAL","amount":61380,"status":"approved","category":"gas"},
    {"id":"M006","date":"2026-07-08","merchant":"PAN PA' YA","amount":16700,"status":"approved","category":"restaurants"},
    {"id":"M007","date":"2026-07-10","merchant":"CLARO HOGAR","amount":112900,"status":"approved","category":"internet"},
    {"id":"M008","date":"2026-07-12","merchant":"ACUEDUCTO DE BOGOTÁ","amount":96750,"status":"approved","category":"water"},
    {"id":"M009","date":"2026-07-12","merchant":"PSE ACUEDUCTO BOGOTÁ","amount":96750,"status":"rejected","category":"water"},
    {"id":"M010","date":"2026-07-14","merchant":"RECARGA NEQUI","amount":30000,"status":"approved","category":"top_up"},
    {"id":"M011","date":"2026-07-17","merchant":"RESTAURANTE CREPES","amount":57800,"status":"approved","category":"restaurants"},
    {"id":"M012","date":"2026-07-19","merchant":"VANTI GAS NATURAL","amount":61380,"status":"rejected","category":"gas"},
    {"id":"M013","date":"2026-07-21","merchant":"NETFLIX","amount":26900,"status":"approved","category":"subscriptions"},
    {"id":"M014","date":"2026-07-23","merchant":"ENVÍO A CAMILA","amount":75000,"status":"approved","category":"transfers"},
    {"id":"M015","date":"2026-07-25","merchant":"TRANSMILENIO","amount":20000,"status":"approved","category":"transport"},
    {"id":"M016","date":"2026-07-28","merchant":"FARMATODO","amount":43250,"status":"approved","category":"health"},
    {"id":"M017","date":"2026-07-30","merchant":"TIENDAS D1","amount":71320,"status":"approved","category":"groceries"},
]
allowed_operations = {"sum", "count", "list"}
allowed_statuses = {"approved", "rejected", "reversed"}
allowed_categories = {tx["category"] for tx in transactions}

def pesos(value):
    return "$" + f"{value:,}".replace(",", ".")

def ledger_text():
    status_es = {"approved":"Aprobada", "rejected":"Rechazada"}
    lines = ["Estas son mis transacciones de julio de 2026:"]
    for tx in transactions:
        lines.append(
            f'{tx["id"]} | {tx["date"]} | {tx["merchant"]} | '
            f'{pesos(tx["amount"])} | {status_es[tx["status"]]}'
        )
    return "\n".join(lines)

def prompt_for(content):
    return tokenizer.apply_chat_template(
        [{"role":"user", "content":content}], tokenize=False,
        add_generation_prompt=True, enable_thinking=True,
    )

def run_trace(content, max_tokens):
    parts, last = [], None
    for event in stream_generate(
        model, tokenizer, prompt=prompt_for(content),
        max_tokens=max_tokens, sampler=make_sampler(temp=0.0),
    ):
        parts.append(event.text)
        last = event
    if last is None:
        raise RuntimeError("The model produced no generation events.")
    return "".join(parts), last

def valid(plan):
    filters = plan.get("filters", {})
    return (
        plan.get("operation") in allowed_operations
        and set(filters.get("categories", [])) <= allowed_categories
        and set(filters.get("status", [])) <= allowed_statuses
        and isinstance(filters.get("date_from"), str)
        and isinstance(filters.get("date_to"), str)
    )

def valid_plans(trace):
    decoder, candidates = json.JSONDecoder(), {}
    for start, char in enumerate(trace):
        if char != "{":
            continue
        try:
            candidate, _ = decoder.raw_decode(trace[start:])
        except json.JSONDecodeError:
            continue
        if isinstance(candidate, dict) and valid(candidate):
            candidates[json.dumps(candidate, sort_keys=True)] = candidate
    return list(candidates.values())

def execute(plan):
    f = plan["filters"]
    selected = [
        tx for tx in transactions
        if f["date_from"] <= tx["date"] <= f["date_to"]
        and tx["category"] in f["categories"]
        and tx["status"] in f["status"]
    ]
    operation = plan["operation"]
    value = (sum(tx["amount"] for tx in selected) if operation == "sum"
             else len(selected) if operation == "count" else selected)
    return {"value":value, "transactions":selected}

def excerpt(trace, head, tail):
    if len(trace) <= head + tail:
        return trace
    omitted = len(trace) - head - tail
    return trace[:head] + f"\n\n... [{omitted} characters omitted] ...\n\n" + trace[-tail:]

# Same natural-language question, path A: unconstrained direct reasoning.
direct_trace_reused = (
    "direct_trace" in globals() and "direct_event" in globals()
    and direct_event.generation_tokens == 4096
    and "TRANSMILENIO" in direct_trace
)
if not direct_trace_reused:
    direct_trace, direct_event = run_trace(ledger_text() + "\n\n" + question, 4096)

# Same natural-language question, path B: reasoning under a plan contract.
contract = f"""
Convierte una pregunta sobre movimientos financieros en un plan JSON ejecutable.
Hoy es 2026-08-19.
Contrato: {{"operation":"<one operation>","filters":{{"date_from":"YYYY-MM-DD","date_to":"YYYY-MM-DD","categories":["..."],"status":["..."]}}}}
Elige operation como exactamente un valor de ["sum", "count", "list"].
Elige cada status únicamente de ["approved", "rejected", "reversed"].
Categorías disponibles: {sorted(allowed_categories)}.
Política del producto: "servicios públicos" significa electricity, gas y water;
internet y top_up son categorías distintas. Un gasto efectivo debe estar approved.
Devuelve el plan JSON después de terminar tu razonamiento. No calcules valores ni selecciones IDs manualmente.
Pregunta: {question}
""".strip()
harness_trace, harness_event = run_trace(contract, 4096)

Path("/kaggle/working/direct_thinking_trace.txt").write_text(direct_trace)
Path("/kaggle/working/harness_thinking_trace.txt").write_text(harness_trace)

candidates = valid_plans(harness_trace)
if len(candidates) != 1:
    raise RuntimeError(f"Expected exactly one valid plan; found {len(candidates)}.")
plan = candidates[0]
result = execute(plan)

print("EXPERIMENT CONTRACT")
print("-------------------")
print("Question reused verbatim: True")
print("Thinking enabled:         True")
print("Decoding:                 greedy / temperature 0")
print("Budget per path:          4096 tokens")
print(f"Direct trace reused:      {direct_trace_reused}")

print("\nDIRECT THINKING METADATA")
print("------------------------")
print(f"Generated tokens:     {direct_event.generation_tokens}")
print(f"Finish reason:        {direct_event.finish_reason}")
print(f"Reasoning closed:     {'</think>' in direct_trace}")
print(f"Final answer found:   {bool(direct_trace.split('</think>',1)[1].strip()) if '</think>' in direct_trace else False}")
print("\nDIRECT TRACE — REPRESENTATIVE EXCERPT")
print("-------------------------------------")
print(excerpt(direct_trace, 3600, 1700))

print("\nHARNESS THINKING METADATA")
print("-------------------------")
print(f"Generated tokens:     {harness_event.generation_tokens}")
print(f"Finish reason:        {harness_event.finish_reason}")
print(f"Reasoning closed:     {'</think>' in harness_trace}")
print(f"Final answer found:   {bool(harness_trace.split('</think>',1)[1].strip()) if '</think>' in harness_trace else False}")
print("\nHARNESS TRACE — REPRESENTATIVE EXCERPT")
print("--------------------------------------")
print(excerpt(harness_trace, 5200, 1400))

print("\nSTRUCTURAL RECOVERY")
print("-------------------")
print(f"Distinct schema-valid plans: {len(candidates)}")
print(json.dumps(plan, ensure_ascii=False, indent=2))
print("\nDETERMINISTIC EXECUTION")
for tx in result["transactions"]:
    print(f'✓ {tx["id"]}  {tx["merchant"]:<24} {pesos(tx["amount"]):>10}')
print(f'\nRECOVERED TOTAL: {pesos(result["value"])}')
print("\nFull traces saved to /kaggle/working/direct_thinking_trace.txt and harness_thinking_trace.txt")


EXPERIMENT CONTRACT
-------------------
Question reused verbatim: True
Thinking enabled:         True
Decoding:                 greedy / temperature 0
Budget per path:          4096 tokens
Direct trace reused:      True

DIRECT THINKING METADATA
------------------------
Generated tokens:     4096
Finish reason:        length
Reasoning closed:     False
Final answer found:   False

DIRECT TRACE — REPRESENTATIVE EXCERPT
-------------------------------------
Thinking Process:

1.  **Analyze the Request:**
    *   Input: A list of transactions for July 2026 (M001 to M017).
    *   Question: "Oye, ¿cuánto me gasté en servicios públicos el mes pasado?" (Hey, how much did I spend on public services last month?)
    *   Context: The user provided a list of transactions with dates, merchant names, amounts, and status (Approved/Rejected).
    *   Task: Identify which transactions correspond to "servicios públicos" (public services) and sum their amounts.

2.  **Analyze the Transaction Data:**
  

### Qué demuestra la comparación con presupuesto equivalente

Al fijar el presupuesto de generación en 4.096 tokens para ambas rutas, se elimina la diferencia de presupuesto como posible explicación del resultado:

- **Ruta directa:** alcanzó el límite de tokens, no completó el bloque `</think>` y no produjo una respuesta final. Su razonamiento observable clasificó erróneamente CLARO HOGAR, RECARGA NEQUI y TRANSMILENIO como servicios públicos y convergió en **COP 488.450**.
- **Ruta con harness:** también alcanzó el mismo límite sin completar `</think>` ni producir una respuesta final. Sin embargo, su traza contenía exactamente un plan distinto y válido según el esquema: `sum`, julio de 2026, categorías `electricity`, `gas` y `water`, estado `approved`.
- **Recuperación estructural:** como $|C(\tau)|=1$, el ejecutor determinista seleccionó M002, M005 y M008 y devolvió **COP 325.550**. Cero o varios planes válidos habrían producido un fallo explícito.

Ninguna ruta terminó, por lo que la diferencia observada no consiste en finalizar antes. En esta ejecución, la traza bajo contrato expuso un único objeto intermedio verificable que el harness pudo ejecutar de forma segura.

La observabilidad convierte el comportamiento del modelo en información operacional. La causa de terminación, la presencia o ausencia de `</think>`, los candidatos estructurados y su validación permiten distinguir una respuesta completa de una generación incompleta o de una traza que ya contiene un resultado utilizable. Cada acción del harness puede rastrearse hasta una señal observable y una regla explícita.

La actualización registrada reutilizó la traza directa de 4.096 tokens ya ejecutada y generó únicamente la traza del harness con presupuesto equivalente. La salida lo documenta explícitamente como `Direct trace reused: True`.

> **El harness no terminó el razonamiento del modelo. Convirtió un estado intermedio observable y verificable en un resultado verificable del sistema.**
